# Overall Results

Set `JOB_FOLDER` in the cell below to a completed job directory. The cell reads `results/_stats_cache.json` and `run.log`, then plots fitness history and the per-Sunday delta distribution against the random solution.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Edit this to point at a completed job folder, not the results folder.
# Example on Windows:
# JOB_FOLDER = Path(r"D:\\Coding\\FAKS\\Mentor\\working-sundays\\.sunday-cache\\jetpans\\d9a3343b-3665-403e-8b11-3c55aa63d381")
JOB_FOLDER = None


def load_stats_cache(job_folder):
    job_folder = Path(job_folder)
    cache_path = job_folder / "results" / "_stats_cache.json"
    if not cache_path.exists():
        raise FileNotFoundError(f"Stats cache not found: {cache_path}")

    payload = json.loads(cache_path.read_text(encoding="utf-8"))
    stats = payload.get("stats", payload)
    return payload, stats


def parse_fitness_history(job_folder):
    job_folder = Path(job_folder)
    run_log_path = job_folder / "run.log"
    if not run_log_path.exists():
        return pd.DataFrame(columns=["step", "iteration", "fitness", "line"])

    iteration_pattern = re.compile(r"Iteration:\s*(\d+)", re.IGNORECASE)
    fitness_pattern = re.compile(
        r"New alpha has fitness of:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)",
        re.IGNORECASE,
    )

    rows = []
    last_iteration = None
    for line_number, line in enumerate(run_log_path.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
        iteration_match = iteration_pattern.search(line)
        if iteration_match:
            last_iteration = int(iteration_match.group(1))

        fitness_match = fitness_pattern.search(line)
        if fitness_match:
            rows.append(
                {
                    "step": len(rows) + 1,
                    "iteration": last_iteration,
                    "fitness": float(fitness_match.group(1)),
                    "line": line_number,
                }
            )

    return pd.DataFrame(rows, columns=["step", "iteration", "fitness", "line"])


def stats_to_sunday_frame(stats):
    rows = []
    for entry in stats.get("per_sunday", []):
        random_value = float(entry.get("random", 0.0))
        optimized_value = float(entry.get("optimized", 0.0))
        delta = float(entry.get("delta", optimized_value - random_value))
        delta_pct = None if random_value == 0 else delta / abs(random_value) * 100.0
        rows.append(
            {
                "sunday": int(entry.get("sunday", len(rows))),
                "random": random_value,
                "optimized": optimized_value,
                "delta": delta,
                "delta_pct": delta_pct,
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df["sunday_label"] = df["sunday"] + 1
    return df


def plot_fitness_history(fitness_df, use_iteration=False, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))

    if fitness_df.empty:
        ax.text(0.5, 0.5, "No fitness history found in run.log", ha="center", va="center")
        ax.set_axis_off()
        return ax

    x_col = "iteration" if use_iteration and fitness_df["iteration"].notna().any() else "step"
    ax.plot(fitness_df[x_col], fitness_df["fitness"], marker="o", linewidth=2, markersize=3)
    ax.set_title("Fitness over optimization progress")
    ax.set_xlabel("Iteration" if x_col == "iteration" else "New-best event")
    ax.set_ylabel("Fitness")
    ax.grid(True, alpha=0.25)
    return ax


def plot_delta_distribution(sunday_df, bins=None, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))

    if sunday_df.empty:
        ax.text(0.5, 0.5, "No Sunday deltas available", ha="center", va="center")
        ax.set_axis_off()
        return ax

    deltas = sunday_df["delta"].to_numpy(dtype=float)
    max_abs = max(1.0, float(np.max(np.abs(deltas))))
    if bins is None:
        bin_count = min(17, max(7, int(np.ceil(np.sqrt(len(deltas)) * 2))))
        if bin_count % 2 == 0:
            bin_count += 1
        bins = np.linspace(-max_abs, max_abs, bin_count + 1)

    ax.hist(deltas, bins=bins, color="#38bdf8", edgecolor="#0f172a", alpha=0.75)
    ax.axvline(0, color="#0f172a", linestyle="--", linewidth=1.5, label="zero")
    ax.axvline(deltas.mean(), color="#16a34a", linestyle="-", linewidth=1.5, label=f"mean={deltas.mean():.3f}")
    ax.set_title("Distribution of per-Sunday delta vs random")
    ax.set_xlabel("Optimized fitness - random fitness")
    ax.set_ylabel("Sunday count")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.25)
    return ax


def plot_delta_by_sunday(sunday_df, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 4))

    if sunday_df.empty:
        ax.text(0.5, 0.5, "No Sunday deltas available", ha="center", va="center")
        ax.set_axis_off()
        return ax

    colors = np.where(sunday_df["delta"] >= 0, "#22c55e", "#f43f5e")
    ax.bar(sunday_df["sunday_label"], sunday_df["delta"], color=colors)
    ax.axhline(0, color="#0f172a", linewidth=1)
    ax.set_title("Per-Sunday delta vs random")
    ax.set_xlabel("Sunday")
    ax.set_ylabel("Optimized fitness - random fitness")
    ax.grid(True, axis="y", alpha=0.25)
    return ax


if JOB_FOLDER is None:
    print("Set JOB_FOLDER to a completed job directory, then run this cell.")
else:
    cache_payload, stats = load_stats_cache(JOB_FOLDER)
    fitness_df = parse_fitness_history(JOB_FOLDER)
    sunday_df = stats_to_sunday_frame(stats)

    print(f"Loaded stats from: {Path(JOB_FOLDER) / 'results' / '_stats_cache.json'}")
    print("Cache metadata:")
    print({k: v for k, v in cache_payload.items() if k != "stats"})
    print("Overall stats:")
    print(stats.get("overall", {}))

    display(sunday_df.head())
    display(fitness_df.head())

    fig, axes = plt.subplots(3, 1, figsize=(12, 12), constrained_layout=True)
    plot_fitness_history(fitness_df, ax=axes[0])
    plot_delta_distribution(sunday_df, ax=axes[1])
    plot_delta_by_sunday(sunday_df, ax=axes[2])
    plt.show()
